In [0]:
%sql
SELECT 1 AS teste;

In [0]:
%sql
DESCRIBE TABLE workspace.silver.empresas;

In [0]:
%sql
DESCRIBE TABLE workspace.silver.cotacoes;

In [0]:
%sql

CREATE OR REPLACE TABLE workspace.gold.dim_empresa AS
SELECT
  -- Gera uma chave substituta sequencial para identificar cada registro da dimensão.
  -- Diferente do id_empresa (chave de negócio), a SK identifica uma versão específica.
  ROW_NUMBER() OVER (ORDER BY id_empresa) AS sk_empresa,

  -- Atributos descritivos e cadastrais das empresas.
  id_empresa,
  ticker, cnpj, cnpj_valido, razao_social, nome_pregao,
  setor, subsetor, segmento_listagem, situacao, uf, cidade,
  data_fundacao, data_listagem, valor_mercado, num_funcionarios,
  site, email_ri, telefone,

  -- Campos de controle do SCD Tipo 2 (histórico de alterações).
  -- Na carga inicial, todas as versões são consideradas vigentes.
  DATE '1900-01-01' AS data_inicio,  -- Início convencional da validade histórica.
  DATE '9999-12-31' AS data_fim,     -- Data futura indica versão ainda aberta.
  true AS flag_atual                -- TRUE identifica a versão atual da empresa.

FROM workspace.silver.empresas;

In [0]:
%sql
SELECT
  COUNT(*)                                    AS linhas,         -- esperado: 450
  COUNT(DISTINCT sk_empresa)                  AS sk_distintas,   -- esperado: 450
  COUNT(DISTINCT id_empresa)                  AS ids_distintos,  -- esperado: 450
  SUM(CASE WHEN flag_atual THEN 1 ELSE 0 END) AS atuais,         -- esperado: 450
  MIN(data_fim)                               AS menor_data_fim  -- esperado: 9999-12-31
FROM workspace.gold.dim_empresa;

In [0]:

%sql
-- Cria a tabela fato de cotações diárias na Gold, utilizando os dados tratados da Silver.
-- Granularidade esperada: uma linha por empresa e data de pregão.
-- Relaciona as cotações à dimensão de empresas pela chave substituta (sk_empresa).

CREATE OR REPLACE TABLE workspace.gold.fato_cotacao_diaria AS
SELECT
  -- Chave substituta da dimensão e data do evento.
  d.sk_empresa,
  c.data_pregao,

  -- Métricas financeiras e operacionais de cada pregão.
  c.preco_abertura, c.preco_maximo, c.preco_minimo, c.preco_fechamento,
  c.volume_financeiro, c.qtd_negocios, c.qtd_titulos,

  -- Calcula o retorno percentual entre o fechamento atual e o anterior disponível.
  -- LAG recupera o preço anterior da mesma empresa.
  -- PARTITION BY separa o cálculo por empresa e ORDER BY organiza por data.
  -- ROUND limita o resultado a quatro casas decimais.
  -- O primeiro registro de cada empresa terá retorno NULL.
  ROUND(
    (c.preco_fechamento /
     LAG(c.preco_fechamento) OVER (
       PARTITION BY c.id_empresa
       ORDER BY c.data_pregao
     ) - 1) * 100,
    4
  ) AS retorno_diario_pct

FROM workspace.silver.cotacoes c

-- INNER JOIN associa cada cotação à versão atual da empresa na dimensão.
-- Os identificadores devem possuir tipos compatíveis e valores padronizados.
JOIN workspace.gold.dim_empresa d
  ON c.id_empresa = d.id_empresa
  AND d.flag_atual;


In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM workspace.silver.cotacoes)                  AS linhas_silver,   -- esperado: 29322
  COUNT(*)                                                          AS linhas_fato,     -- esperado: 29322 (igual)
  COUNT(DISTINCT sk_empresa)                                        AS empresas,        -- esperado: 450
  SUM(CASE WHEN retorno_diario_pct IS NULL THEN 1 ELSE 0 END)       AS retornos_nulos,  -- esperado: 450
  ROUND(MIN(retorno_diario_pct), 2)                                 AS menor_retorno,   -- plausível: perto de -10 a -20
  ROUND(MAX(retorno_diario_pct), 2)                                 AS maior_retorno    -- plausível: perto de +10 a +20
FROM workspace.gold.fato_cotacao_diaria;

In [0]:
%sql
-- O volume total não pode ter mudado entre silver e fato (esperado: diferença perto de 0)
SELECT ROUND(
  (SELECT SUM(volume_financeiro) FROM workspace.silver.cotacoes)
  - (SELECT SUM(volume_financeiro) FROM workspace.gold.fato_cotacao_diaria), 2) AS diferenca_volume;

In [0]:

%sql
-- Cria indicadores mensais de cotação na Gold, com uma linha por empresa e mês.
-- Utiliza os dados diários da tabela fato para calcular retorno, quantidade de pregões e volumes.

CREATE OR REPLACE TABLE workspace.gold.kpi_retorno_mensal AS

-- CTE base: identifica o primeiro e o último pregão disponível de cada empresa no mês.
WITH base AS (
  SELECT
    sk_empresa,
    date_format(data_pregao, 'yyyy-MM') AS ano_mes,
    preco_fechamento,
    volume_financeiro,

    -- Numera os pregões em ordem crescente para identificar o primeiro fechamento.
    ROW_NUMBER() OVER (
      PARTITION BY sk_empresa, date_format(data_pregao, 'yyyy-MM')
      ORDER BY data_pregao ASC
    ) AS rn_ini,

    -- Numera em ordem decrescente para identificar o último fechamento.
    ROW_NUMBER() OVER (
      PARTITION BY sk_empresa, date_format(data_pregao, 'yyyy-MM')
      ORDER BY data_pregao DESC
    ) AS rn_fim

  FROM workspace.gold.fato_cotacao_diaria
),

-- CTE mensal: consolida os indicadores por empresa e mês.
mensal AS (
  SELECT
    sk_empresa,
    ano_mes,

    -- Recupera o preço de fechamento do primeiro e do último pregão do mês.
    MAX(CASE WHEN rn_ini = 1 THEN preco_fechamento END) AS fechamento_inicial,
    MAX(CASE WHEN rn_fim = 1 THEN preco_fechamento END) AS fechamento_final,

    -- Calcula quantidade de registros, volume total e volume médio por pregão.
    COUNT(*) AS qtd_pregoes,
    SUM(volume_financeiro) AS volume_total,
    AVG(volume_financeiro) AS volume_medio_diario

  FROM base
  GROUP BY sk_empresa, ano_mes
)

-- Calcula a variação percentual entre o primeiro e o último fechamento disponível.
-- ROUND limita o resultado a quatro casas decimais.
-- Não inclui a variação entre o mês anterior e o primeiro pregão deste mês.
SELECT
  *,
  ROUND((fechamento_final / fechamento_inicial - 1) * 100, 4) AS retorno_mensal_pct
FROM mensal;


In [0]:
%sql
SELECT
  COUNT(*)                            AS linhas,        -- esperado: 1350 (450 empresas x 3 meses)
  COUNT(DISTINCT sk_empresa)          AS empresas,      -- esperado: 450
  COUNT(DISTINCT ano_mes)             AS meses,         -- esperado: 3
  SUM(qtd_pregoes)                    AS soma_pregoes,  -- esperado: 29322 (todas as linhas da fato)
  ROUND(MIN(retorno_mensal_pct), 2)   AS menor,         -- sem valores absurdos (nada além de +-100)
  ROUND(MAX(retorno_mensal_pct), 2)   AS maior
FROM workspace.gold.kpi_retorno_mensal;

In [0]:

%sql
-- Cria um ranking de empresas por valor de mercado dentro de cada setor.
-- Calcula também a participação percentual de cada empresa no valor total do setor.
-- Considera apenas as versões atuais das empresas (SCD Tipo 2).

CREATE OR REPLACE TABLE workspace.gold.ranking_setor_valor_mercado AS
SELECT
  sk_empresa,
  id_empresa,
  setor,
  razao_social,
  valor_mercado,

  -- RANK classifica as empresas do maior para o menor valor de mercado.
  -- PARTITION BY reinicia o ranking para cada setor.
  -- DESC ordena do maior para o menor; NULLS LAST mantém valores nulos no final.
  -- Empresas empatadas recebem a mesma posição no ranking.
  RANK() OVER (
    PARTITION BY setor
    ORDER BY valor_mercado DESC NULLS LAST
  ) AS rank_no_setor,

  -- Calcula a participação percentual da empresa no valor total do setor.
  -- SUM OVER calcula o total por setor sem agrupar ou eliminar linhas.
  -- ROUND arredonda o percentual para duas casas decimais.
  -- Empresas com valor de mercado NULL terão participação NULL.
  ROUND(
    valor_mercado /
    SUM(valor_mercado) OVER (PARTITION BY setor) * 100,
    2
  ) AS pct_do_setor

FROM workspace.gold.dim_empresa

-- Considera apenas a versão vigente de cada empresa.
WHERE flag_atual;


In [0]:
%sql
SELECT
  COUNT(*)                                                AS linhas,        -- esperado: 450
  COUNT(DISTINCT setor)                                   AS setores,       -- esperado: 11
  SUM(CASE WHEN rank_no_setor = 1 THEN 1 ELSE 0 END)      AS lideres        -- esperado: 11 (um por setor)
FROM workspace.gold.ranking_setor_valor_mercado;

In [0]:
%sql
-- Consulta os líderes de valor de mercado de cada setor na camada Gold.
-- WHERE rank_no_setor = 1 seleciona apenas as empresas na primeira posição de seu setor.
-- Em caso de empate, RANK permite que mais de uma empresa apareça como líder.
-- ORDER BY DESC organiza os líderes do maior para o menor valor de mercado.

SELECT
  setor,
  razao_social,
  valor_mercado,
  pct_do_setor
FROM workspace.gold.ranking_setor_valor_mercado
WHERE rank_no_setor = 1
ORDER BY valor_mercado DESC;

In [0]:
%sql
-- Exibe todas as empresas do ranking setorial, com todas as colunas.
SELECT *
FROM workspace.gold.ranking_setor_valor_mercado
ORDER BY setor, rank_no_setor

In [0]:

%sql
-- Cria um resumo consolidado por setor na camada Gold.
-- Combina indicadores cadastrais das empresas com o retorno mensal médio.
-- Granularidade: uma linha por setor econômico.

CREATE OR REPLACE TABLE workspace.gold.resumo_por_setor AS

-- CTE cadastro: agrega informações das empresas por setor.
WITH cadastro AS (
  SELECT
    setor,

    -- COUNT conta todas as empresas, mesmo com atributos nulos.
    COUNT(*) AS qtd_empresas,

    -- SUM calcula os totais e AVG calcula a média do valor de mercado.
    -- Valores NULL são ignorados nas somas e médias.
    SUM(valor_mercado) AS valor_mercado_total,
    AVG(valor_mercado) AS valor_mercado_medio,
    SUM(num_funcionarios) AS total_funcionarios

  FROM workspace.gold.dim_empresa

  -- Considera apenas as versões vigentes das empresas (SCD Tipo 2).
  WHERE flag_atual

  -- Consolida os indicadores por setor.
  GROUP BY setor
),

-- CTE retorno: calcula a média simples dos retornos mensais por setor.
retorno AS (
  SELECT
    d.setor,

    -- AVG calcula a média dos retornos empresa-mês disponíveis.
    -- ROUND limita o resultado a quatro casas decimais.
    ROUND(AVG(k.retorno_mensal_pct), 4) AS retorno_mensal_medio_pct

  FROM workspace.gold.kpi_retorno_mensal k

  -- JOIN recupera o setor de cada empresa pela chave substituta.
  JOIN workspace.gold.dim_empresa d
    ON k.sk_empresa = d.sk_empresa

  GROUP BY d.setor
)

-- LEFT JOIN preserva todos os setores cadastrados.
-- Setores sem retorno disponível permanecem na tabela com retorno NULL.
SELECT
  c.*,
  r.retorno_mensal_medio_pct

FROM cadastro c
LEFT JOIN retorno r
  ON c.setor = r.setor;


In [0]:
%sql
SELECT COUNT(*) AS setores, SUM(qtd_empresas) AS total_empresas   -- esperado: 11 e 450
FROM workspace.gold.resumo_por_setor;

In [0]:
%sql
-- Consulta todos os indicadores consolidados por setor na camada Gold.
-- SELECT * retorna todas as colunas da tabela.
-- ORDER BY DESC organiza os setores do maior para o menor valor de mercado total.
-- Permite comparar quantidade de empresas, valor de mercado, funcionários e retorno médio.

SELECT *
FROM workspace.gold.resumo_por_setor
ORDER BY valor_mercado_total DESC;

In [0]:
%sql
SELECT 'dim_empresa' AS tabela, COUNT(*) AS linhas FROM workspace.gold.dim_empresa                       -- 450
UNION ALL SELECT 'fato_cotacao_diaria',        COUNT(*) FROM workspace.gold.fato_cotacao_diaria          -- 29322
UNION ALL SELECT 'kpi_retorno_mensal',         COUNT(*) FROM workspace.gold.kpi_retorno_mensal           -- 1350
UNION ALL SELECT 'ranking_setor_valor_mercado',COUNT(*) FROM workspace.gold.ranking_setor_valor_mercado  -- 450
UNION ALL SELECT 'resumo_por_setor',           COUNT(*) FROM workspace.gold.resumo_por_setor;            -- 11

Decisões de modelagem — Camada Gold

Chave substituta (Surrogate Key): a dimensão dim_empresa utiliza sk_empresa para identificar cada versão de uma empresa, preparando a estrutura para o histórico SCD Tipo 2.

Vigência histórica: na carga inicial, data_inicio = 1900-01-01 e data_fim = 9999-12-31 representam uma versão inicial aberta. Essas datas são convenções de modelagem, não datas reais das alterações.

Relacionamento fato-dimensão: a fato_cotacao_diaria relaciona as cotações à dimensão por id_empresa e armazena sk_empresa. Os identificadores devem estar padronizados na Silver para garantir correspondência no JOIN.

Retorno diário: calculado com LAG() sobre o fechamento anterior disponível da mesma empresa. Registros em quarentena podem gerar intervalos maiores entre as cotações utilizadas.

Retorno mensal: calculado pela variação entre o primeiro e o último fechamento disponível no mês, identificados por ROW_NUMBER() em ordem crescente e decrescente. Não considera a variação entre o fechamento do mês anterior e o primeiro pregão do mês atual.

Tratamento de valores nulos: empresas sem valor de mercado são mantidas no ranking com NULLS LAST, preservando a cobertura cadastral. A participação percentual dessas empresas permanece NULL.

Consolidação setorial: os indicadores são agregados por setor, utilizando LEFT JOIN para preservar setores sem dados de retorno disponíveis.
